In [16]:
!pip install contractions

In [17]:
import kagglehub
import pandas as pd
import re
import contractions
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [18]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [19]:
path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/uciml/sms-spam-collection-dataset


In [20]:
import os

print(os.listdir(path))

['spam.csv']


In [21]:
df = pd.read_csv(
    os.path.join(path, "spam.csv"),
    encoding="latin-1"
)

df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [22]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns)

Shape: (5572, 5)

Columns:
Index(['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='object')


In [23]:
df = df[["v1", "v2"]]

df.columns = ["label", "message"]

df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [24]:
rows, columns = df.shape

print("Number of rows:", rows)
print("Number of columns:", columns)

Number of rows: 5572
Number of columns: 2


In [25]:
print(df.isna().sum())

label      0
message    0
dtype: int64


In [26]:
df = df.dropna(subset=["message"])

print("Shape after removing missing values:", df.shape)

Shape after removing missing values: (5572, 2)


In [27]:
duplicate_count = df["message"].duplicated().sum()

print("Duplicate messages:", duplicate_count)

Duplicate messages: 403


In [28]:
df = df.drop_duplicates(subset=["message"])

print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (5169, 2)


In [29]:
# Block 14: Check label distribution
print(df["label"].value_counts())

label
ham     4516
spam     653
Name: count, dtype: int64


In [30]:
# Block 15: Display sample message
print("Sample message:")
print(df["message"].iloc[0])

Sample message:
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...


In [31]:
# Block 16: Initialize stopwords and lemmatizer
stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()

In [32]:
# Block 17: Create text preprocessing function
def preprocess_text(text):

    # 1. Expand contractions
    text = contractions.fix(text)

    # 2. Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # 3. Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # 4. Convert text to lowercase
    text = text.lower()

    # 5. Remove numbers and special characters
    text = re.sub(r"[^a-zA-Z\s]", "", text)

    # 6. Tokenization
    tokens = word_tokenize(text)

    # 7. Remove stopwords
    tokens = [
        token for token in tokens
        if token not in stop_words
    ]

    # 8. Lemmatization
    tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
    ]

    # 9. Join tokens back into text
    return " ".join(tokens)

In [33]:
# Block 18: Test preprocessing on one message
sample_message = df["message"].iloc[0]

print("BEFORE PREPROCESSING:")
print(sample_message)

print("\n" + "=" * 80)

print("AFTER PREPROCESSING:")
print(preprocess_text(sample_message))

BEFORE PREPROCESSING:
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

AFTER PREPROCESSING:
go jurong point crazy available bugis n great world la e buffet cine got amore wat


In [34]:
# Block 19: Apply preprocessing to the complete dataset
df["cleaned_message"] = df["message"].apply(preprocess_text)

In [35]:
# Block 20: Display processed dataset
df.head()

,label,message,cleaned_message
0,ham,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,ham,Ok lar... Joking wif u oni...,ok lar joking wif oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win fa cup final tkts st ...
3,ham,U dun say so early hor... U c already then say...,dun say early hor c already say
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah think go usf life around though


In [36]:
# Block 21: Compare original and cleaned message
print("ORIGINAL MESSAGE:")
print(df["message"].iloc[0])

print("\n" + "=" * 100)

print("CLEANED MESSAGE:")
print(df["cleaned_message"].iloc[0])

ORIGINAL MESSAGE:
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

CLEANED MESSAGE:
go jurong point crazy available bugis n great world la e buffet cine got amore wat


In [37]:
# Block 22: Test contraction expansion
example_text = "I didn't receive the message because it wasn't sent."

print("Before:")
print(example_text)

print("\nAfter:")
print(contractions.fix(example_text))

Before:
I didn't receive the message because it wasn't sent.

After:
I did not receive the message because it was not sent.


In [38]:
# Block 23: Tokenization example
sentence = "I received an amazing message today."

tokens = word_tokenize(sentence)

print(tokens)

['I', 'received', 'an', 'amazing', 'message', 'today', '.']


In [39]:
# Block 24: Stopword removal example
sentence = "I received an amazing message today."

tokens = word_tokenize(sentence.lower())

print("Before:")
print(tokens)

filtered_tokens = [
    token for token in tokens
    if token not in stop_words
]

print("\nAfter:")
print(filtered_tokens)

Before:
['i', 'received', 'an', 'amazing', 'message', 'today', '.']

After:
['received', 'amazing', 'message', 'today', '.']


In [40]:
# Block 25: Lemmatization example
words_to_lemmatize = [
    "playing",
    "played",
    "plays",
    "messages"
]

for word in words_to_lemmatize:
    print(word, "→", lemmatizer.lemmatize(word))

playing → playing
played → played
plays → play
messages → message


In [41]:
# Block 26: Compare word counts before and after preprocessing
df["words_before"] = df["message"].apply(
    lambda text: len(word_tokenize(text))
)

df["words_after"] = df["cleaned_message"].apply(
    lambda text: len(word_tokenize(text))
)

In [42]:
# Block 27: Display word count comparison
df[
    [
        "message",
        "cleaned_message",
        "words_before",
        "words_after"
    ]
].head()

,message,cleaned_message,words_before,words_after
0,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...,24,16
1,Ok lar... Joking wif u oni...,ok lar joking wif oni,8,5
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win fa cup final tkts st ...,37,20
3,U dun say so early hor... U c already then say...,dun say early hor c already say,13,7
4,"Nah I don't think he goes to usf, he lives aro...",nah think go usf life around though,15,7


In [43]:
# Block 28: Save preprocessed dataset
df.to_csv(
    "/kaggle/working/SMS_Spam_preprocessed.csv",
    index=False
)
# Block 29: Confirmation
print("Preprocessed SMS Spam dataset saved successfully!")

Preprocessed SMS Spam dataset saved successfully!
